# E-Commerce Lakehouse: Bronze Customers

## Ingest and validate source customers

Read the original customer CSV with all source fields as strings.
Preserve leading zeros in ZIP code prefixes and add ingestion metadata.
Save the Bronze table and compare source and saved row counts.

## Ingest and refresh Bronze customers

Read the complete customer CSV, preserving source values as strings
and retaining ZIP prefix leading zeros. Add ingestion metadata.

Verify 99,441 source records before replacing the Bronze table,
then confirm that the saved row count matches.

In [0]:
from pyspark.sql.functions import current_timestamp

raw_customers_df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/ecommerce/raw_files/olist_customers_dataset.csv")
)

source_count = raw_customers_df.count()
print("Source customer rows:", source_count)

if source_count != 99441:
    raise ValueError(
        f"Expected 99,441 customer records; found {source_count}. "
        "Stopped before replacing the table."
    )

bronze_customers_df = raw_customers_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

table_name = "workspace.ecommerce.bronze_customers"

(
    bronze_customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_bronze_customers_df = spark.table(table_name)
saved_count = saved_bronze_customers_df.count()

if saved_count != source_count:
    raise ValueError("Saved Bronze customer count does not match the source.")

print("Bronze customers refreshed:", saved_count)

Source customer rows: 99441
Bronze customers refreshed: 99441


In [0]:
print("Source customer rows:", raw_customers_df.count())
raw_customers_df.printSchema()

Source customer rows: 99441
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

